# FPO Training on Mujoco Environments

This notebook clones the FPO repository, installs dependencies, trains FPO on Ant, Humanoid, Hopper, Walker, and HalfCheetah environments, and displays training metrics and graphs for rewards and episode lengths.

## 1. Clone the Repository

In [ ]:
!git clone https://github.com/manavparikh2203/fpo.git /kaggle/working/fpo
%cd /kaggle/working/fpo

## 2. Install Dependencies

In [ ]:
!cd playground && pip install -e .
!pip install "gymnasium[mujoco]" optax tqdm

In [ ]:
import sys
sys.path.insert(0, '/kaggle/working/fpo/playground/src')

## 3. Set Up the Environment

In [ ]:
import os
os.environ['MUJOCO_GL'] = 'egl'
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

In [ ]:
import gymnasium as gym

env = gym.make('Ant-v5')
obs, info = env.reset()

print('Gymnasium Ant-v5 created successfully')
env.close()

print('Observation shape:', obs.shape)
print('Action space:', env.action_space)

## 4. Run the Training Script

In [ ]:
# Final save
pd.DataFrame(history).to_csv(output_dir / "train_metrics.csv", index=False)
import pickle
with open(output_dir / "agent_state.pkl", "wb") as f:
    pickle.dump(agent_state, f)
print(f"Training complete. Results saved to {output_dir}")

In [ ]:
import pickle
from pathlib import Path
import gymnasium as gym
from gym.wrappers import RecordVideo
import jax.numpy as jnp
import numpy as np

output_dir = Path('./gym_fpo_results') / 'Antv5'

# Load trained agent
with open(output_dir / 'agent_state.pkl', 'rb') as f:
    agent_state = pickle.load(f)

# Record video
env = gym.make('Ant-v5', render_mode='rgb_array')
env = RecordVideo(env, video_folder=str(output_dir), episode_trigger=lambda x: True)
obs, _ = env.reset()
for _ in range(1000):
    action, _ = agent_state.sample_action(jnp.asarray(obs, dtype=jnp.float32), jax.random.key(0), deterministic=True)
    obs, reward, terminated, truncated, _ = env.step(np.asarray(action))
    if terminated or truncated:
        break
env.close()
print("Video saved to", output_dir)

# Check success
import pandas as pd
df = pd.read_csv(output_dir / 'train_metrics.csv')
final_reward = df['mean_episode_reward'].iloc[-1]
if final_reward > 1000:
    print(f"Training successful! Final mean episode reward: {final_reward:.2f}")
else:
    print(f"Training may need more iterations. Final mean episode reward: {final_reward:.2f}")

**Note:** The Gymnasium Ant-v5 FPO training is handled by the previous cell running `train_fpo_gymnasium_ant.py`. The old playground batch script is not used for this Gymnasium-only workflow.

## 5. Display Training Metrics

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

output_dir = Path('./gym_fpo_results') / 'Antv5'

train_csv = output_dir / 'train_metrics.csv'
if train_csv.exists():
    df = pd.read_csv(train_csv)
    print('Gymnasium Ant-v5 FPO training metrics:')
    print(df.head())
    print('\n')
else:
    print('No training metrics found at', train_csv)

## 6. Plot Graphs

In [ ]:
from IPython.display import Image, display

output_dir = Path('./gym_fpo_results') / 'Antv5'

reward_png = output_dir / 'reward_curve.png'
if reward_png.exists():
    display(Image(filename=str(reward_png)))
else:
    print('No reward plot found at', reward_png)